# trapX LLM Advisory — **`advisory_any_bar.py`** (any date + bar)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_any_bar_colab_v2.ipynb)

**Latest version.** This notebook runs the *actual* standalone `advisory_any_bar.py` tool — not a re-implementation. Given a **DATE** and **BAR_TIME** it:

1. Downloads that day's folder from the shared Google Drive (`gdown`, public, no Drive API) — the `llm_advisory_<date>.jsonl`, the trapX SQLite checkpoint, and the day CSVs.
2. **Gates** on the jsonl: stops if no pattern fired that minute.
3. Rebuilds the advisory input fresh (state-memory @ bar−1, the engine-computed snapshot per fired touchpoint, the minute's market data).
4. Fires **one converged LLM-advisory call** and prints the verdict.

The script, all skill prompts, **and a minimal `project/` package** (the engine functions + self-contained backbone modules — jerk/signal backbones, jerk_type, touchpoint_horizon, jerk_genuineness, skill_trace — that the script imports at runtime) are **embedded below** (base64) — the notebook is fully self-contained.

**Backends:** `nvidia` (default, needs `NVIDIA_API_KEY`) or `anthropic` / `auto` (live-parity, needs `ANTHROPIC_API_KEY`).

## 1. Install deps + load API keys
Keys are read from Colab **Secrets** (🔑 left sidebar → add `NVIDIA_API_KEY` and/or `ANTHROPIC_API_KEY`, toggle notebook access). Set at least the one matching your chosen backend.

In [ ]:
!pip install -q gdown openai anthropic python-dotenv langgraph langgraph-checkpoint-sqlite

import os
try:
    from google.colab import userdata
    for _k in ('NVIDIA_API_KEY', 'ANTHROPIC_API_KEY'):
        try:
            _v = userdata.get(_k)
        except Exception:
            _v = None
        if _v:
            os.environ[_k] = _v.strip()
except Exception:
    pass

print('NVIDIA_API_KEY   :', 'set' if os.environ.get('NVIDIA_API_KEY') else 'MISSING')
print('ANTHROPIC_API_KEY:', 'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING')

## 2. Write the embedded `advisory_any_bar.py` + `skills/` + `project/` to disk
Decodes the base64 payloads into `/content/`. The minimal `project/` package carries the two AST-sliced engine functions (`trapx_agent._audit_compute_v5_flags`, `llm_advisory.advisory._build_opening_audit_user_message`) — which let the script **recompute the v5 layer** (the recorded jsonl carries no `v5_*` fields; the live engine logs raw facts, v5 is derived) — **plus the self-contained `llm_advisory` backbone modules** (jerk/signal backbones, jerk_type, touchpoint_horizon, jerk_genuineness, skill_trace) the converged advisory path imports.

In [ ]:
import base64, json, pathlib

SCRIPT_B64 = "..."
SKILLS_B64 = "..."
PROJECT_B64 = "..."

pathlib.Path('advisory_any_bar.py').write_bytes(base64.b64decode(SCRIPT_B64))
skills = json.loads(base64.b64decode(SKILLS_B64).decode('utf-8'))
sd = pathlib.Path('skills'); sd.mkdir(exist_ok=True)
for name, text in skills.items():
    (sd / name).write_text(text, encoding='utf-8')
proj = json.loads(base64.b64decode(PROJECT_B64).decode('utf-8'))
for relpath, text in proj.items():
    p = pathlib.Path(relpath); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8')
print('wrote advisory_any_bar.py +', len(skills), 'skills +', len(proj),
      'project files (engine v5 recompute enabled)')

## 3. Choose the bar + backend + model
`WHEN` is the bar in the script's native `DD-mon, HH:MM` form. `BACKEND`: `nvidia` (default), `anthropic`, or `auto` (pins to the backend/model the live engine recorded — like-for-like replay).

**`MODEL`** is the NVIDIA model id passed straight to `--model`. Leave it **blank** for the live-engine parity model (`meta/llama-3.3-70b-instruct`), or pick another from the dropdown — e.g. **`z-ai/glm-5.1`** — or type any id your NVIDIA gateway serves. (Changing the model changes the verdict — it is no longer engine-parity.)

In [ ]:
#@title Parameters { run: "auto" }
WHEN    = "12-Jun, 09:19"  #@param {type:"string"}
BACKEND = "nvidia"         #@param ["nvidia", "anthropic", "auto"]
YEAR    = 2026             #@param {type:"integer"}
# NVIDIA model id. "" = backend default (meta/llama-3.3-70b-instruct, the
# live-engine parity model). Pick from the dropdown or type any gateway id.
MODEL   = ""               #@param ["", "z-ai/glm-5.1", "meta/llama-3.3-70b-instruct"] {allow-input: true}
print('WHEN   =', WHEN)
print('BACKEND=', BACKEND, '  YEAR=', YEAR,
      '  MODEL=', MODEL or '(default: meta/llama-3.3-70b-instruct)')

## 4. Mount Google Drive + resolve the day folder
Mounts your Drive and resolves the **day folder** for the chosen bar (`VM-4-logs/<Mon_DD>`, e.g. `Jun_12`). The notebook reuses the script's own date parser so the folder name always matches.

In [ ]:
#@title Drive folder { run: "auto" }
VM_LOGS_DIR = "/content/drive/MyDrive/VM-4-logs"  #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob, argparse
sys.path.insert(0, '.')
import advisory_any_bar as aab           # the script we wrote in step 2

# Reuse the script's exact parser so the Drive folder name always matches.
req = aab.parse_request(argparse.Namespace(when=WHEN, date=None,
                                           time=None, year=YEAR))
FOLDER_NAME = req.mon_dd                  # e.g. 'Jun_12'
DAY_DIR  = f'{VM_LOGS_DIR}/{FOLDER_NAME}'
LOG_NAME = f"advisory_{req.yyyymmdd}_{req.time.replace(':', '')}.log"
LOG_OUT  = f'{DAY_DIR}/{LOG_NAME}'

os.makedirs(DAY_DIR, exist_ok=True)
HAS_LOCAL = bool(glob.glob(f'{DAY_DIR}/llm_advisory_*.jsonl'))
print(f'date {req.iso_date}  ->  Drive folder {FOLDER_NAME!r}')
print('DAY_DIR :', DAY_DIR)
print('source  :', 'Drive day folder (--local-dir)' if HAS_LOCAL
      else 'gdown download (folder has no jsonl yet)')
print('log_out :', LOG_OUT)

## 5. Run the advisory — log saved back to the Drive folder
Runs the tool via a tiny `run_advisory.py` wrapper (written below) with `--skills-dir skills --backend <BACKEND> --no-live --local-dir <DAY_DIR> --out <DAY_DIR>/advisory_<date>_<time>.log`. The subprocess runs from `/content`, so it imports the embedded `project/` package and recomputes the v5 layer.

> **`--no-live` is required on Colab.** For *today*'s date the script switches to a *live* path that reads a local Postgres only present on the trapX VM. `--no-live` forces the Drive/CSV replay path.

> **Why the wrapper?** On Colab there is **no Postgres**, and the script's `pg_connect()` raises `SystemExit` on a failed connect — which its replay CSV-only fallback (`except Exception`) cannot catch, so the run would die with exit 1 *before the gate*. The wrapper turns that `SystemExit` into a catchable error so the script's **intended `conn=None` CSV-only path** runs (every verdict input is in the Drive CSVs; only the rarely-material run-cumulative TRAP is skipped — and it's reported, not silent). `advisory_any_bar.py` itself is **unchanged**.

In [ ]:
import subprocess, sys, shlex, pathlib, textwrap

# Colab has no Postgres. pg_connect() raises SystemExit on a failed connect,
# but the script's replay CSV-only fallback only does `except Exception` —
# and SystemExit is NOT an Exception, so the process would die (exit 1) before
# the jsonl gate. This wrapper converts that SystemExit into a catchable error
# so the intended conn=None CSV-only path runs. advisory_any_bar.py UNCHANGED.
WRAPPER = textwrap.dedent('''
    import sys
    import advisory_any_bar as aab
    _orig = aab.pg_connect
    def _pg_optional():
        try:
            return _orig()
        except BaseException as e:
            raise RuntimeError("postgres unavailable on Colab; CSV-only: %s" % e)
    aab.pg_connect = _pg_optional
    sys.exit(aab.main())
''')
pathlib.Path('run_advisory.py').write_text(WRAPPER, encoding='utf-8')

cmd = [sys.executable, 'run_advisory.py', WHEN,
       '--skills-dir', 'skills', '--backend', BACKEND, '--year', str(YEAR),
       '--no-live', '--out', LOG_OUT]
if HAS_LOCAL:
    cmd += ['--local-dir', DAY_DIR]
if MODEL.strip():
    cmd += ['--model', MODEL.strip()]
print('$', ' '.join(shlex.quote(c) for c in cmd), '\n')

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stderr)
print('=' * 70)
print(proc.stdout)
if proc.returncode != 0:
    print('\n[exit code]', proc.returncode)

## 6. Confirm the log on Drive + show it
Lists the verbose log(s) now in the Drive day folder and prints the newest.

In [ ]:
import glob, os
logs = sorted(glob.glob(f'{DAY_DIR}/advisory_{req.yyyymmdd}_*.log'),
              key=os.path.getmtime)
if logs:
    print(f'{len(logs)} verbose log(s) saved in Drive folder {DAY_DIR}:')
    for p in logs:
        print('   ', os.path.basename(p), f'({os.path.getsize(p)} bytes)')
    print('\n===== newest:', os.path.basename(logs[-1]), '=====\n')
    print(open(logs[-1], encoding='utf-8').read())
else:
    print('no verbose log in', DAY_DIR,
          '— did the run gate out (no pattern at that bar) or error?')